# Tema: Fundamentos físicos y operaciones de Delta

## Objetivos
Crear tablas, cargar y sustituir datos, inspeccionar versiones y relacionar las transacciones con archivos.

## Conceptos importantes para el examen
Delta combina datos Parquet y registro transaccional _delta_log. ACID: atomicidad, consistencia, aislamiento y durabilidad. Un commit publica una nueva versión; UPDATE no equivale necesariamente a editar archivos en sitio. Las deletion vectors pueden representar cambios sin reescritura inmediata.

**Dificultad:** Básico · **Tiempo estimado:** 75 min.

El acceso directo a archivos de tablas managed de UC está restringido. La inspección física utiliza una copia Delta por ruta en un volumen, sin registrarla como tabla externa.

Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_04_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
employees = spark.createDataFrame(
    [(i, f"Empleado {i:02d}", ["Data", "Sales", "Finance"][i % 3],
      30000 + i * 1500, i % 4 != 0,
      datetime(2026, 1, 1), datetime(2026, 1, 1)) for i in range(1, 19)],
    "employee_id INT, name STRING, department STRING, salary INT, active BOOLEAN, created_at TIMESTAMP, updated_at TIMESTAMP"
)
employees.createOrReplaceTempView("employees_seed")
employees.write.format("delta").mode("errorifexists").saveAsTable("employees")
display(employees.orderBy("employee_id"))

In [ ]:
# Requiere CREATE VOLUME en el schema; alternativa: usa un volumen autorizado.
spark.sql("CREATE VOLUME IF NOT EXISTS lab_files")
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/lab_files"
dbutils.fs.mkdirs(BASE + "/landing")
CHECKPOINT = BASE + "/checkpoints/main"
print(BASE)

## PARTE 1 - EJEMPLOS GUIADOS

### 1. CREATE, USING DELTA e INSERT
Cada sentencia de escritura confirma su propia transacción en este laboratorio.

In [ ]:
%sql
CREATE TABLE employees_demo (employee_id INT, name STRING, salary INT) USING DELTA;
INSERT INTO employees_demo SELECT employee_id, name, salary FROM employees;
DESCRIBE HISTORY employees_demo;

### 2. CTAS e INSERT OVERWRITE
CTAS obtiene tipos de la consulta. OVERWRITE sustituye los datos actuales; el historial sigue existiendo.

In [ ]:
%sql
CREATE TABLE employees_copy USING DELTA AS SELECT * FROM employees_demo;
INSERT OVERWRITE employees_copy SELECT * FROM employees_demo WHERE salary >= 45000;
DESCRIBE DETAIL employees_copy;

### 3. DML e historial

In [ ]:
%sql
UPDATE employees_demo SET salary = salary + 100 WHERE employee_id = 1;
DELETE FROM employees_demo WHERE employee_id = 2;
DESCRIBE HISTORY employees_demo;

### 4. Inspección física
No cuentes filas leyendo directamente todos los Parquet: incluirías archivos obsoletos o filas eliminadas lógicamente. El lector Delta interpreta el log.

In [ ]:
physical_path = BASE + "/delta_physical"
employees.write.format("delta").mode("errorifexists").save(physical_path)
display(dbutils.fs.ls(physical_path))
display(dbutils.fs.ls(physical_path + "/_delta_log"))
spark.sql(f"UPDATE delta.`{physical_path}` SET salary = salary + 1 WHERE employee_id = 1")
display(spark.sql(f"DESCRIBE HISTORY delta.`{physical_path}`"))

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Crea practice_employees con CTAS a partir de employees. Inspecciona formato, ubicación, número de archivos y tamaño.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Consulta el historial antes y después de insertar el empleado 99 copiando el resto de campos del 1.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Actualiza el salario de 99 a 60.000 y elimina el 2. Muestra qué operaciones aparecen en el historial.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Sustituye el contenido de practice_employees por los activos originales; compara recuento e historial.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Crea una tabla pequeña con CLUSTER BY (department), ejecuta OPTIMIZE y consulta el detalle. Explica por qué aquí no demuestras una mejora de rendimiento.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** DESCRIBE DETAIL.

**Pista 2:** INSERT SELECT cambiando la clave.

**Pista 3:** Dos escrituras son dos commits.

**Pista 4:** INSERT OVERWRITE no es DROP.

**Pista 5:** Liquid clustering organiza archivos; 18 filas no sirven de benchmark.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
%sql
CREATE OR REPLACE TABLE practice_employees USING DELTA AS SELECT * FROM employees;
DESCRIBE DETAIL practice_employees;

### Solución 2

In [ ]:
%sql
DESCRIBE HISTORY practice_employees;
INSERT INTO practice_employees SELECT 99, name, department, salary, active, created_at, updated_at FROM employees WHERE employee_id = 1;
DESCRIBE HISTORY practice_employees;

### Solución 3

In [ ]:
%sql
UPDATE practice_employees SET salary = 60000 WHERE employee_id = 99;
DELETE FROM practice_employees WHERE employee_id = 2;
DESCRIBE HISTORY practice_employees;

### Solución 4

In [ ]:
%sql
INSERT OVERWRITE practice_employees SELECT * FROM employees WHERE active;
SELECT COUNT(*) AS n FROM practice_employees;
DESCRIBE HISTORY practice_employees;

### Solución 5

In [ ]:
%sql
CREATE OR REPLACE TABLE employees_clustered USING DELTA CLUSTER BY (department) AS SELECT * FROM employees;
OPTIMIZE employees_clustered;
DESCRIBE DETAIL employees_clustered;

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
¿Qué define qué archivos pertenecen a la versión actual?

A. La fecha del directorio

B. El registro transaccional Delta

C. El nombre más corto

D. Todos los Parquet de la ruta

### Pregunta 2
¿Qué garantiza atomicidad?

A. Que nunca haya NULL

B. Que no exista latencia

C. Que un commit sea visible completo o no sea visible

D. Que cada tabla tenga un solo archivo

### Pregunta 3
¿Qué hace INSERT OVERWRITE sin particiones?

A. Sustituye el contenido actual con la consulta

B. Añade filas siempre

C. Elimina necesariamente todo el historial

D. Convierte Delta a CSV

### Respuestas y explicación
**1. B** — El log registra las acciones del snapshot.

**2. C** — No se publican escrituras parcialmente confirmadas.

**3. A** — La sustitución queda registrada como una nueva versión.

### Documentación oficial
- [Delta](https://docs.databricks.com/aws/en/delta/)
- [Liquid clustering](https://docs.databricks.com/aws/en/delta/clustering)

## PARTE 6 - RETO FINAL
Diseña una secuencia CREATE → INSERT → UPDATE → DELETE → OVERWRITE y una tabla de observaciones con versión, operación y recuento. Explica por qué el número de archivos no coincide necesariamente con el número de commits.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
